In [ ]:
# FASE 5 — EVALUATION | CardioRisk · IBM Data Science · CRISP-DM
# Evaluación clínica completa del mejor modelo en el conjunto de TEST (datos no vistos)

!pip install kagglehub imbalanced-learn xgboost shap --quiet

import kagglehub, os, warnings, joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.facecolor'] = '#0a0f1a'
matplotlib.rcParams['axes.facecolor']   = '#0d1526'
matplotlib.rcParams['text.color']       = '#e2e8f0'
matplotlib.rcParams['axes.labelcolor']  = '#e2e8f0'
matplotlib.rcParams['xtick.color']      = '#7a8fa8'
matplotlib.rcParams['ytick.color']      = '#7a8fa8'
matplotlib.rcParams['axes.edgecolor']   = '#1a2c3d'
matplotlib.rcParams['grid.color']       = '#1a2c3d'
warnings.filterwarnings('ignore')
print("✓ Librerías cargadas")

In [ ]:
# ── BLOQUE 1: RECONSTRUIR PIPELINE (autocontenido) ──
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE

KAGGLE_DATASET = "jocelyndumlao/cardiovascular-disease-dataset"
try:
    path = kagglehub.dataset_download(KAGGLE_DATASET)
    csv_path = next(f for f in [os.path.join(r,f) for r,_,fs in os.walk(path) for f in fs] if f.endswith('.csv'))
    df = pd.read_csv(csv_path)
except:
    df = pd.read_csv('/content/cardiovascular_disease_dataset.csv')

df.columns = df.columns.str.lower().str.strip()
df = df.dropna()

CATEGORICAL = ['gender','chestpain','restingelectro']
TARGET      = 'target'
df_enc      = pd.get_dummies(df, columns=CATEGORICAL, drop_first=True)
feature_cols = [c for c in df_enc.columns if c != TARGET]
X = df_enc[feature_cols]
y = df_enc[TARGET]

X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.15, random_state=42, stratify=y)
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.1765, random_state=42, stratify=y_temp)

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

sm = SMOTE(random_state=42)
X_train_sm, y_train_sm = sm.fit_resample(X_train_sc, y_train)

# Cargar modelo (si viene de F4) o reentrenar
try:
    best_model = joblib.load('/content/best_model.pkl')
    print("✓ Modelo cargado desde F4")
except:
    from sklearn.ensemble import RandomForestClassifier
    best_model = RandomForestClassifier(n_estimators=400, random_state=42)
    best_model.fit(X_train_sm, y_train_sm)
    print("✓ Modelo reentrenado (Random Forest fallback)")

y_pred = best_model.predict(X_test_sc)
y_prob = best_model.predict_proba(X_test_sc)[:,1]
print(f"\nEvaluación sobre TEST SET ({len(y_test)} pacientes no vistos)")

In [ ]:
# ── BLOQUE 2: MÉTRICAS COMPLETAS ──
from sklearn.metrics import (recall_score, precision_score, f1_score,
                              roc_auc_score, accuracy_score,
                              confusion_matrix, classification_report)

recall    = recall_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
f1        = f1_score(y_test, y_pred)
auc       = roc_auc_score(y_test, y_prob)
accuracy  = accuracy_score(y_test, y_pred)
cm        = confusion_matrix(y_test, y_pred)

tn, fp, fn, tp = cm.ravel()
especificidad = tn / (tn + fp)

print("═" * 48)
print(f"  RESULTADOS FINALES — TEST SET")
print("═" * 48)
print(f"  Recall (Sensibilidad) : {recall:.4f}  ← MÉTRICA PRIMARIA")
print(f"  Precision             : {precision:.4f}")
print(f"  F1-Score              : {f1:.4f}")
print(f"  AUC-ROC               : {auc:.4f}")
print(f"  Accuracy              : {accuracy:.4f}")
print(f"  Especificidad         : {especificidad:.4f}")
print("─" * 48)
print(f"  Verdaderos Positivos  : {tp}  (alto riesgo detectado ✓)")
print(f"  Falsos Negativos      : {fn}  (alto riesgo NO detectado ✗)")
print(f"  Falsos Positivos      : {fp}  (bajo riesgo clasificado como alto)")
print(f"  Verdaderos Negativos  : {tn}  (bajo riesgo detectado ✓)")
print("═" * 48)
print(f"\n{classification_report(y_test, y_pred, target_names=['Bajo Riesgo','Alto Riesgo'])}")


In [ ]:
# ── BLOQUE 3: MATRIZ DE CONFUSIÓN ──
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(cm, cmap='Blues', alpha=.8)
ax.set_xticks([0,1]); ax.set_yticks([0,1])
ax.set_xticklabels(['Bajo Riesgo','Alto Riesgo'])
ax.set_yticklabels(['Bajo Riesgo','Alto Riesgo'])
ax.set_xlabel('Predicción'); ax.set_ylabel('Real')
ax.set_title('Matriz de Confusión — Test Set', fontsize=13)

for i in range(2):
    for j in range(2):
        color = '#ff3355' if (i==1 and j==0) else '#00f2fe'
        ax.text(j, i, str(cm[i,j]), ha='center', va='center',
                fontsize=22, fontweight='bold', color=color)

labels = [['TN','FP'],['FN ✗','TP ✓']]
for i in range(2):
    for j in range(2):
        ax.text(j, i+.32, labels[i][j], ha='center', va='center', fontsize=9, color='#7a8fa8')

plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.savefig('/content/f5_confusion_matrix.png', dpi=150, bbox_inches='tight', facecolor='#0a0f1a')
plt.show()

In [ ]:
# ── BLOQUE 4: CURVA ROC + PRECISION-RECALL ──
from sklearn.metrics import roc_curve, precision_recall_curve, average_precision_score

fpr, tpr, _ = roc_curve(y_test, y_prob)
prec_c, rec_c, _ = precision_recall_curve(y_test, y_prob)
ap = average_precision_score(y_test, y_prob)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# ROC
axes[0].plot(fpr, tpr, color='#00f2fe', lw=2, label=f'AUC = {auc:.3f}')
axes[0].plot([0,1],[0,1],'--',color='#3a5570',lw=1)
axes[0].fill_between(fpr, tpr, alpha=.07, color='#00f2fe')
axes[0].set_xlabel('FPR'); axes[0].set_ylabel('TPR (Recall)')
axes[0].set_title('Curva ROC'); axes[0].legend(); axes[0].grid(alpha=.3)

# Precision-Recall
axes[1].plot(rec_c, prec_c, color='#8b5cf6', lw=2, label=f'AP = {ap:.3f}')
axes[1].axhline(y_test.mean(), color='#3a5570', linestyle='--', lw=1, label='Baseline')
axes[1].fill_between(rec_c, prec_c, alpha=.07, color='#8b5cf6')
axes[1].set_xlabel('Recall'); axes[1].set_ylabel('Precision')
axes[1].set_title('Curva Precision-Recall'); axes[1].legend(); axes[1].grid(alpha=.3)

plt.suptitle('Evaluación del Modelo — Curvas de Rendimiento', fontsize=13)
plt.tight_layout()
plt.savefig('/content/f5_roc_pr_curves.png', dpi=150, bbox_inches='tight', facecolor='#0a0f1a')
plt.show()

In [ ]:
# ── BLOQUE 5: IMPORTANCIA DE VARIABLES / SHAP ──
import shap

try:
    explainer = shap.TreeExplainer(best_model)
    shap_vals = explainer.shap_values(X_test_sc)
    vals = shap_vals[1] if isinstance(shap_vals, list) else shap_vals

    fig, ax = plt.subplots(figsize=(9, 6))
    mean_shap = np.abs(vals).mean(axis=0)
    order = np.argsort(mean_shap)[-12:]
    feat_names = [str(f) for f in feature_cols]
    ax.barh([feat_names[i] for i in order], mean_shap[order],
            color='#00f2fe', alpha=.75, edgecolor='none')
    ax.set_xlabel('|SHAP value| promedio')
    ax.set_title('Importancia de Variables (SHAP)', fontsize=13)
    ax.grid(axis='x', alpha=.3)
    plt.tight_layout()
    plt.savefig('/content/f5_shap_importance.png', dpi=150, bbox_inches='tight', facecolor='#0a0f1a')
    plt.show()
    print("✓ SHAP ejecutado correctamente")
except Exception as e:
    print(f"SHAP no disponible ({e}) — usando feature_importances_")
    if hasattr(best_model, 'feature_importances_'):
        imp = pd.Series(best_model.feature_importances_, index=feature_cols).sort_values()
        fig, ax = plt.subplots(figsize=(9,6))
        imp[-12:].plot(kind='barh', ax=ax, color='#00f2fe', alpha=.75)
        ax.set_title('Importancia de Variables (Gini)', fontsize=13)
        ax.grid(axis='x', alpha=.3)
        plt.tight_layout()
        plt.savefig('/content/f5_feature_importance.png', dpi=150, bbox_inches='tight', facecolor='#0a0f1a')
        plt.show()

print(f"\n→ FASE 5 COMPLETA")
print(f"   Recall final (test): {recall:.4f}")
print(f"   AUC-ROC  final: {auc:.4f}")
kpi_ok = "✓ KPI ALCANZADO" if recall >= 0.80 else f"⚠ KPI objetivo: 0.80 | Obtenido: {recall:.3f}"
print(f"   {kpi_ok}")